# MSMARCO-XI RAG Indexing — Kaggle Stable Build

## Architecture

**Hugging Face Hub API → remote Parquet streaming → PyArrow row groups → chunking → Transformer embeddings → FAISS → benchmark**

This notebook is deliberately designed to avoid the two failures seen in the earlier versions:

1. **No `datasets.load_dataset(..., streaming=True)`** for MSMARCO-XI, avoiding the nested Arrow conversion failure.
2. **No `pip install -U numpy/pandas/scipy/scikit-learn` and no `sentence-transformers` import**, avoiding Kaggle's NumPy/SciPy binary incompatibility.

The dataset repository contains language-specific Parquet files such as `train/hintrain.parquet`. The schema contains `query`, `Answer`, `query_id`, `query_type`, nested `passages` with `is_selected`, `English_passages`, and `Translated_passages`, plus `Eng_Query` and `Eng_Answer`.

### Kaggle requirements
- Internet: **ON**
- Accelerator: **GPU**
- Kaggle Secret: optional `HF_TOKEN` (recommended)
- Do **not** attach or download the full 55 GB dataset.


## 0. Environment safety check

Run this first. It does not modify Kaggle's core NumPy/Pandas/SciPy stack.


In [ ]:
import sys
import importlib.util
import subprocess

required = {
    "huggingface_hub": "huggingface_hub",
    "pyarrow": "pyarrow",
    "transformers": "transformers",
    "faiss": "faiss-cpu",
    "fsspec": "fsspec",
}

missing = [
    pip_name
    for module, pip_name in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing only missing packages:", missing)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--no-deps", "-q", *missing
    ])
else:
    print("All required modules are already available.")

import numpy as np
import pandas as pd
import scipy
import sklearn
import torch

print("\nEnvironment:")
print("Python :", sys.version.split()[0])
print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)
print("SciPy  :", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("Torch  :", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))


## 1. Hugging Face authentication

For this public dataset, the token is **not technically required**. A read token is still useful for stable authenticated Hub access and model downloads.

Create a Hugging Face **read** token in your HF account, then store it in Kaggle:

**Kaggle → Add-ons → Secrets → Add secret**
- Label: `HF_TOKEN`
- Value: your `hf_...` token

Never hard-code the token in this notebook.


In [ ]:
import os

HF_TOKEN = os.environ.get("HF_TOKEN")

# Kaggle Secrets API
try:
    from kaggle_secrets import UserSecretsClient
    secret_client = UserSecretsClient()
    HF_TOKEN = secret_client.get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    if HF_TOKEN:
        print("HF_TOKEN loaded from environment.")
    else:
        print("HF_TOKEN not found. Public dataset access may still work.")

if HF_TOKEN:
    assert HF_TOKEN.startswith("hf_"), "The HF token format looks incorrect."


## 2. Dataset configuration

To avoid unnecessary transfer, start with **one language**. Hindi is selected here.

Set `MAX_ROWS` to a small number for the first test. Once the pipeline is proven, set it higher or to `None`.

**Important:** `STREAM_REMOTE_PARQUET=True` means the Parquet file is accessed through an HTTP range-capable filesystem. It is **not copied wholesale into `/kaggle/working`**.


In [ ]:
from pathlib import Path

LANGUAGE = "hi"
SPLIT = "train"

PASSAGE_MODE = "translated"   # translated | english

MAX_ROWS = 20_000             # first successful test; later increase
BATCH_ROWS = 512

CHUNK_STRATEGY = "metadata"   # fixed | sentence | semantic | metadata

FIXED_CHARS = 500
OVERLAP_CHARS = 80
SENTENCES_PER_CHUNK = 3
SEMANTIC_THRESHOLD = 0.58

TOP_K = 5

STREAM_REMOTE_PARQUET = True

OUTPUT_DIR = Path("/kaggle/working/msmarco_xi_artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

prefix = {
    "as": "asm",
    "bn": "ben",
    "gu": "guj",
    "hi": "hin",
    "kn": "kan",
    "ml": "mal",
    "mr": "mar",
    "ne": "nep",
    "or": "ori",
    "pa": "pan",
    "sa": "san",
    "ta": "tam",
    "te": "tel",
    "ur": "urd",
}[LANGUAGE]

file_name = f"{prefix}train.parquet" if SPLIT == "train" else f"{prefix}val.parquet"
repo_file = f"{SPLIT}/{file_name}"

print("Dataset :", "ai4bharat/MSMARCO-XI")
print("File    :", repo_file)
print("Rows    :", MAX_ROWS)


## 3. Resolve the remote file through the Hugging Face Hub API

We do not manually construct a `/resolve/main/...` URL.

The Hub API returns the correct remote location for the exact file.


In [ ]:
from huggingface_hub import HfApi, hf_hub_url

api = HfApi(token=HF_TOKEN)

info = api.repo_info(
    repo_id="ai4bharat/MSMARCO-XI",
    repo_type="dataset",
)

print("Repository:", info.id)
print("Revision  :", info.sha)

REMOTE_URL = hf_hub_url(
    repo_id="ai4bharat/MSMARCO-XI",
    filename=repo_file,
    repo_type="dataset",
    revision="main",
)

print("Remote URL resolved by HF Hub:")
print(REMOTE_URL)


## 4. Open Parquet remotely — no full 3.7 GB download

The original dataset file is several GB per language. This cell opens the remote object through HTTP range requests and lets PyArrow read only the required row groups/batches.

If your Kaggle environment cannot range-read the Hugging Face object, the code stops with a clear message instead of silently downloading the entire file.


In [ ]:
import fsspec
import pyarrow.parquet as pq

storage_options = {}
if HF_TOKEN:
    storage_options["headers"] = {"Authorization": f"Bearer {HF_TOKEN}"}

fs = fsspec.filesystem(
    "https",
    **storage_options,
)

remote_file = fs.open(REMOTE_URL, "rb", block_size=8 * 1024 * 1024, cache_type="readahead")

parquet = pq.ParquetFile(remote_file)

print("✅ Remote Parquet opened.")
print("Row groups :", parquet.num_row_groups)
print("Rows       :", parquet.metadata.num_rows)
print("Columns    :", parquet.schema_arrow.names)


## 5. Inspect one record

This reads only a tiny amount from the remote Parquet file.


In [ ]:
first_batch = next(parquet.iter_batches(batch_size=1))
first_record = first_batch.to_pylist()[0]

print("Top-level fields:")
print(list(first_record.keys()))

print("\nquery:", str(first_record.get("query"))[:300])
print("Answer:", str(first_record.get("Answer"))[:300])
print("query_id:", first_record.get("query_id"))
print("query_type:", first_record.get("query_type"))

passages = first_record.get("passages") or {}
print("\nPassage fields:", list(passages.keys()))

for key, value in passages.items():
    if isinstance(value, list):
        print(f"{key}: {len(value)} items")
        if value:
            print("  sample:", str(value[0])[:250])


## 6. Normalize MSMARCO-XI passages


In [ ]:
def normalize_record(record, passage_mode="translated"):
    passages = record.get("passages") or {}

    if passage_mode == "translated":
        texts = passages.get("Translated_passages") or []
    elif passage_mode == "english":
        texts = passages.get("English_passages") or []
    else:
        raise ValueError("PASSAGE_MODE must be 'translated' or 'english'")

    selected = passages.get("is_selected") or []

    rows = []

    for idx, text in enumerate(texts):
        text = str(text).strip()
        if not text:
            continue

        rows.append({
            "query_id": int(record.get("query_id", 0)),
            "query": str(record.get("query", "")),
            "answer": str(record.get("Answer", "")),
            "query_type": str(record.get("query_type", "")),
            "source_lang": str(record.get("source_lang", "")),
            "target_lang": str(record.get("target_lang", "")),
            "passage_index": idx,
            "is_selected": int(selected[idx]) if idx < len(selected) else 0,
            "text": text,
        })

    return rows


normalized = normalize_record(first_record, PASSAGE_MODE)

print("Normalized passages:", len(normalized))
if normalized:
    print(normalized[0])


## 7. Chunking functions

We will benchmark these rather than claiming one is best in advance.


In [ ]:
import re

def split_sentences(text):
    parts = re.split(r"(?<=[.!?।॥])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]


def fixed_chunks(text, size=500, overlap=80):
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + size, len(text))
        piece = text[start:end].strip()

        if piece:
            chunks.append(piece)

        if end >= len(text):
            break

        start += size - overlap

    return chunks


def sentence_chunks(text, sentences_per_chunk=3):
    sentences = split_sentences(text)

    return [
        " ".join(sentences[i:i + sentences_per_chunk])
        for i in range(0, len(sentences), sentences_per_chunk)
    ]


def metadata_chunks(row):
    base = sentence_chunks(row["text"], SENTENCES_PER_CHUNK)

    return [
        (
            f"[lang={row['target_lang']} "
            f"type={row['query_type']} "
            f"query={row['query']}] {chunk}"
        )
        for chunk in base
    ]


print("Chunking utilities ready.")


## 8. Transformer embedding model — no SentenceTransformers

We use Hugging Face Transformers + PyTorch directly.

This avoids importing the `sentence_transformers → sklearn → scipy → numpy` chain that broke the earlier Kaggle environment.


In [ ]:
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "intfloat/multilingual-e5-small"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
).to(device)

model.eval()

print("Model loaded:", MODEL_NAME)
print("Device:", device)


In [ ]:
@torch.inference_mode()
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.inference_mode()
def encode_texts(texts, batch_size=128, prefix="passage: "):
    vectors = []

    for start in range(0, len(texts), batch_size):
        batch = [prefix + x for x in texts[start:start + batch_size]]

        tokens = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )

        tokens = {k: v.to(device) for k, v in tokens.items()}

        output = model(**tokens)

        embeddings = mean_pool(
            output.last_hidden_state,
            tokens["attention_mask"],
        )

        embeddings = torch.nn.functional.normalize(
            embeddings,
            p=2,
            dim=1,
        )

        vectors.append(embeddings.cpu().numpy().astype("float32"))

    return np.vstack(vectors)


# Smoke test before any dataset indexing
test_vectors = encode_texts(
    ["This is a test passage.", "This is another test passage."],
    batch_size=2,
)

print("Embedding shape:", test_vectors.shape)


## 9. Chunking smoke test

Do not start a huge run until this finishes.


In [ ]:
for strategy in ["fixed", "sentence", "metadata"]:
    row = normalized[0] if normalized else None

    if not row:
        continue

    if strategy == "fixed":
        chunks = fixed_chunks(row["text"], FIXED_CHARS, OVERLAP_CHARS)
    elif strategy == "sentence":
        chunks = sentence_chunks(row["text"], SENTENCES_PER_CHUNK)
    else:
        chunks = metadata_chunks(row)

    print(strategy, "=>", len(chunks), "chunks")


## 10. Build a memory-bounded FAISS index

We process the remote Parquet in small batches:

`remote Parquet → rows → chunks → GPU embeddings → FAISS → metadata file`

Nothing accumulates to the size of the entire corpus.


In [ ]:
import json
import numpy as np
import faiss

INDEX_PATH = OUTPUT_DIR / "msmarco_xi.faiss"
METADATA_PATH = OUTPUT_DIR / "metadata.jsonl"
CONFIG_PATH = OUTPUT_DIR / "config.json"

dimension = int(test_vectors.shape[1])

index = faiss.IndexFlatIP(dimension)

metadata_file = open(METADATA_PATH, "w", encoding="utf-8")

pending_texts = []
pending_metadata = []

records_processed = 0
vectors_indexed = 0

def make_chunks(row):
    if CHUNK_STRATEGY == "fixed":
        chunks = fixed_chunks(
            row["text"],
            FIXED_CHARS,
            OVERLAP_CHARS,
        )
    elif CHUNK_STRATEGY == "sentence":
        chunks = sentence_chunks(
            row["text"],
            SENTENCES_PER_CHUNK,
        )
    elif CHUNK_STRATEGY == "metadata":
        chunks = metadata_chunks(row)
    else:
        raise ValueError(
            "Use fixed, sentence, or metadata for the first stable build."
        )

    return [
        {
            "text": chunk,
            "query_id": row["query_id"],
            "query": row["query"],
            "answer": row["answer"],
            "query_type": row["query_type"],
            "source_lang": row["source_lang"],
            "target_lang": row["target_lang"],
            "passage_index": row["passage_index"],
            "chunk_index": i,
            "is_selected": row["is_selected"],
        }
        for i, chunk in enumerate(chunks)
    ]


def flush_batch():
    global pending_texts, pending_metadata, vectors_indexed

    if not pending_texts:
        return

    vectors = encode_texts(
        pending_texts,
        batch_size=128,
        prefix="passage: ",
    )

    index.add(vectors)

    for meta in pending_metadata:
        metadata_file.write(
            json.dumps(meta, ensure_ascii=False) + "\n"
        )

    vectors_indexed += len(pending_texts)

    pending_texts = []
    pending_metadata = []


## 11. Index the first 20,000 records

This is the first real run. If it completes successfully, increase `MAX_ROWS`.

For the first run, do not change anything else.


In [ ]:
import time
from tqdm.auto import tqdm

started = time.perf_counter()

for batch in tqdm(
    parquet.iter_batches(
        batch_size=BATCH_ROWS,
    ),
    desc="Reading remote Parquet",
):
    records = batch.to_pylist()

    for record in records:
        if MAX_ROWS is not None and records_processed >= MAX_ROWS:
            break

        records_processed += 1

        for row in normalize_record(record, PASSAGE_MODE):
            for item in make_chunks(row):
                pending_texts.append(item["text"])
                pending_metadata.append(item)

                if len(pending_texts) >= 128:
                    flush_batch()

    if MAX_ROWS is not None and records_processed >= MAX_ROWS:
        break

flush_batch()
metadata_file.close()

elapsed = time.perf_counter() - started

faiss.write_index(index, str(INDEX_PATH))

config = {
    "dataset": "ai4bharat/MSMARCO-XI",
    "remote_file": repo_file,
    "language": LANGUAGE,
    "split": SPLIT,
    "passage_mode": PASSAGE_MODE,
    "chunk_strategy": CHUNK_STRATEGY,
    "embedding_model": MODEL_NAME,
    "records_processed": records_processed,
    "vectors_indexed": int(index.ntotal),
    "build_seconds": elapsed,
}

CONFIG_PATH.write_text(
    json.dumps(config, indent=2),
    encoding="utf-8",
)

print("✅ INDEX COMPLETE")
print("Records:", records_processed)
print("Vectors:", index.ntotal)
print("Build seconds:", round(elapsed, 2))


## 12. Retrieval benchmark

The competition asks for P50/P70/P100 latency across many queries. This cell measures query embedding + FAISS retrieval.

The **final competition latency** must still be measured end-to-end after adding STT, orchestration, guardrails, and generation.


In [ ]:
# Load metadata for this validation run.
metadata_rows = []

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        metadata_rows.append(json.loads(line))


def retrieve(query, top_k=5):
    q = encode_texts(
        [query],
        batch_size=1,
        prefix="query: ",
    )

    scores, ids = index.search(q, top_k)

    results = []

    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue

        item = metadata_rows[int(idx)].copy()
        item["score"] = float(score)
        results.append(item)

    return results


query_for_test = normalized[0]["query"]

results = retrieve(query_for_test, TOP_K)

print("Query:", query_for_test)

for i, result in enumerate(results, 1):
    print(f"\n#{i} score={result['score']:.4f}")
    print(result["text"][:500])


In [ ]:
# Build a small query set from the processed metadata.
queries = []
seen = set()

for row in metadata_rows:
    q = row["query"].strip()

    if q and q not in seen:
        queries.append(q)
        seen.add(q)

    if len(queries) >= 100:
        break

latencies = []

for q in queries:
    t0 = time.perf_counter()
    retrieve(q, TOP_K)
    latencies.append((time.perf_counter() - t0) * 1000)

if latencies:
    arr = np.asarray(latencies)

    print(f"P50  : {np.percentile(arr, 50):.2f} ms")
    print(f"P70  : {np.percentile(arr, 70):.2f} ms")
    print(f"P100 : {np.max(arr):.2f} ms")
    print(f"Mean : {np.mean(arr):.2f} ms")


## 13. Grounding gate

The final RAG harness should not answer when retrieval is weak.

This is a first gating mechanism; the final threshold must be tuned on validation queries.


In [ ]:
def grounded_retrieval(query, top_k=5, min_score=0.35):
    results = retrieve(query, top_k)

    if not results:
        return {
            "answerable": False,
            "reason": "no_results",
            "results": [],
        }

    best_score = results[0]["score"]

    if best_score < min_score:
        return {
            "answerable": False,
            "reason": "low_retrieval_score",
            "best_score": best_score,
            "results": results,
        }

    return {
        "answerable": True,
        "best_score": best_score,
        "results": results,
    }


print(grounded_retrieval(query_for_test))


## 14. Final artifact summary

The notebook produces:
- FAISS index
- JSONL metadata
- configuration JSON

Do not commit a multi-GB index into GitHub. Keep the source code/config in GitHub and store large index artifacts separately.


In [ ]:
for path in [INDEX_PATH, METADATA_PATH, CONFIG_PATH]:
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"{path.name}: {size_mb:.2f} MB")
